# Assignment 3: Time Series Prediction of Sepsis

Authors:  
Max Masuch  
Ismail Mohammed

## Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas import read_csv
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout, Input
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, multilabel_confusion_matrix, accuracy_score, roc_auc_score

## Load data

This section should load the raw dataset for the task.  
Remember to use relative paths to load any files in the notebook.

In [138]:
df_a = pd.read_csv('raw_data/sepsisexp_timeseries_partition-A.tsv', sep='\t')

df_b = pd.read_csv('raw_data/sepsisexp_timeseries_partition-B.tsv', sep='\t')

df_c = pd.read_csv('raw_data/sepsisexp_timeseries_partition-C.tsv', sep='\t')

df_d = pd.read_csv('raw_data/sepsisexp_timeseries_partition-D.tsv', sep='\t')

print(df_a.head())

      id  sepsis  severity  timestep  respiratory_minute_volume  heart_rate  \
0  12292       0       0.0       0.0                   0.190898    0.424464   
1  12292       0       0.0       0.5                   0.157654    0.667394   
2  12292       0       0.0       1.0                   0.024678    0.618808   
3  12292       0       0.0       1.5                  -0.208030    0.278706   
4  12292       0       0.0       2.0                  -0.108298   -0.352912   

   leukocytes  temperature  partial_co2  respiratory_rate  ...   calcium  \
0    0.301015    -0.168117    -0.275272          1.879692  ...  1.019083   
1    0.301015    -0.168117    -0.275272          1.708485  ...  1.019083   
2    0.301015    -0.732387     1.003408          2.050899  ... -0.868157   
3    0.301015    -0.732387     1.003408          1.366071  ... -0.868157   
4    0.301015    -0.732387     1.094023          1.537278  ... -0.868157   

   potassium  mixed_venous_oxygen_saturation  urine_output  net bala

In [139]:
train_df = pd.concat([df_a, df_b], axis=0).reset_index(drop=True)
val_df = df_c.reset_index(drop=True)
test_df = df_d.reset_index(drop=True)

In [140]:
excluded = ['id', 'sepsis', 'severity', 'timestep']
features = [col for col in train_df.columns if col not in excluded]

In [141]:
def add_targets(df):
    df['target_2h'] = df.groupby('id')['sepsis'].shift(-4)
    df['target_4h'] = df.groupby('id')['sepsis'].shift(-8)
    df['target_6h'] = df.groupby('id')['sepsis'].shift(-12)
    return df

In [142]:
df_train = add_targets(train_df).ffill().fillna(0)
df_val = add_targets(val_df).ffill().fillna(0)
df_test = add_targets(test_df).ffill().fillna(0)

In [143]:
def create_sequences(df, feature_cols, target_col, window_size):
    X, y = [], []
    for _, group in df.groupby('id'):
        if len(group) <= window_size:
            continue
        
        data = group[feature_cols].values
        target = group[target_col].values
        
        for i in range(len(group) - window_size):
            X.append(data[i : i + window_size])
            y.append(target[i + window_size])
            
    return np.array(X), np.array(y)

In [163]:
X_train_2h, y_train_2h = create_sequences(df_train, features, 'target_2h', 12)
X_val_2h, y_val_2h = create_sequences(df_val, features, 'target_2h', 12)
X_test_2h, y_test_2h = create_sequences(df_test, features, 'target_2h', 12)

X_train_4h, y_train_4h = create_sequences(df_train, features, 'target_4h', 24)
X_val_4h, y_val_4h = create_sequences(df_val, features, 'target_4h', 24)
X_test_4h, y_test_4h = create_sequences(df_test, features, 'target_4h', 24)

X_train_6h, y_train_6h = create_sequences(df_train, features, 'target_6h', 36)
X_val_6h, y_val_6h = create_sequences(df_val, features, 'target_6h', 36)
X_test_6h, y_test_6h = create_sequences(df_test, features, 'target_6h', 36)

## DENNA MODEL ÄR HEMSK MAX. IDK WHAT TO DO

In [ ]:
def train_model(X_train, y_train, X_val, y_val):
    
    model = Sequential()

    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

    model.add(LSTM(64, return_sequences=False))

    model.add(Dropout(0.2))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    model.compile(optimizer=optimizer, 
        loss='binary_crossentropy', 
        metrics=['accuracy', 'auc'])
    
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3, 
        restore_best_weights=True)


    model.fit(
        X_train, y_train, 
        validation_data=(X_val, y_val), 
        epochs=25, batch_size=20, 
        callbacks=[early_stop], verbose=1)
    
    return model

In [ ]:
print('2h Model')
model_2h = train_model(X_train_2h, y_train_2h, X_val_2h, y_val_2h)

In [ ]:
print('4h Model')
model_4h = train_model(X_train_4h, y_train_4h, X_val_4h, y_val_4h)

4h Model
Epoch 1/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 148s 10ms/step - accuracy: 0.9300 - auc: 0.9818 - loss: 0.4090 - val_accuracy: 0.7002 - val_auc: 0.7112 - val_loss: 1.2831
Epoch 2/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 159s 11ms/step - accuracy: 0.9734 - auc: 0.9900 - loss: 0.2277 - val_accuracy: 0.6960 - val_auc: 0.7139 - val_loss: 1.3062
Epoch 3/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 159s 11ms/step - accuracy: 0.9797 - auc: 0.9926 - loss: 0.1806 - val_accuracy: 0.6849 - val_auc: 0.7011 - val_loss: 1.4439
Epoch 4/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 158s 11ms/step - accuracy: 0.9824 - auc: 0.9936 - loss: 0.1585 - val_accuracy: 0.6886 - val_auc: 0.7076 - val_loss: 1.4523


In [ ]:
print('6h Model')
model_6h = train_model(X_train_6h, y_train_6h, X_val_6h, y_val_6h)

6h Model
Epoch 1/25
13963/13963 ━━━━━━━━━━━━━━━━━━━━ 280s 20ms/step - accuracy: 0.9459 - auc: 0.9813 - loss: 0.3745 - val_accuracy: 0.6763 - val_auc: 0.7131 - val_loss: 1.1624
Epoch 2/25
13963/13963 ━━━━━━━━━━━━━━━━━━━━ 255s 18ms/step - accuracy: 0.9783 - auc: 0.9898 - loss: 0.1990 - val_accuracy: 0.6870 - val_auc: 0.6930 - val_loss: 1.4110
Epoch 3/25
13963/13963 ━━━━━━━━━━━━━━━━━━━━ 238s 17ms/step - accuracy: 0.9849 - auc: 0.9930 - loss: 0.1487 - val_accuracy: 0.6711 - val_auc: 0.6851 - val_loss: 1.6101
Epoch 4/25
13963/13963 ━━━━━━━━━━━━━━━━━━━━ 280s 20ms/step - accuracy: 0.9855 - auc: 0.9936 - loss: 0.1425 - val_accuracy: 0.6952 - val_auc: 0.6948 - val_loss: 1.5658


In [ ]:
X_test_2h, y_test_2h = create_sequences(df_test, features, 'target_2h', 12)
X_test_4h, y_test_4h = create_sequences(df_test, features, 'target_4h', 24)
X_test_6h, y_test_6h = create_sequences(df_test, features, 'target_6h', 36)

In [162]:
results_2h = model_2h.evaluate(X_test_2h, y_test_2h, verbose=1)
results_4h = model_4h.evaluate(X_test_4h, y_test_4h, verbose=1)
results_6h = model_6h.evaluate(X_test_6h, y_test_6h, verbose=1)

4178/4178 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6422 - auc: 0.6767 - loss: 1.0934
4059/4059 ━━━━━━━━━━━━━━━━━━━━ 28s 7ms/step - accuracy: 0.6237 - auc: 0.6510 - loss: 1.4915
3940/3940 ━━━━━━━━━━━━━━━━━━━━ 39s 10ms/step - accuracy: 0.6341 - auc: 0.6343 - loss: 1.3379


In [ ]:
def get_feature_importance(model, X, y, feature_names):
    baseline_auc = roc_auc_score(y, model.predict(X, verbose=0))
    importances = []
    
    for i in range(len(feature_names)):
        save_col = X[:, :, i].copy()
        X[:, :, i] = np.random.permutation(X[:, :, i])
        
        permuted_auc = roc_auc_score(y, model.predict(X, verbose=0))
        importances.append(baseline_auc - permuted_auc)
        
        X[:, :, i] = save_col
        
    return pd.Series(importances, index=feature_names).sort_values(ascending=False)

importance_2h = get_feature_importance(model_2h, X_test_2h, y_test_2h, features)
print("\nTopp 10 viktigaste särdrag för 2h:")
print(importance_2h.head(10))

## Task 1: This is the title of task 1

This section should contain the solution of task 1.

It is mandatory to maintain the headings for each task.  
OPTIONALLY, you can use one level down (###) to organize subsessions of the assignments.

Use markdown cells like this one to include:
- Discussion points.
- References to specific sources of code that you might have used to solve the assignment.
- General commentas and explanations about your solution.

In [ ]:
# Always use comments in the code to document specific steps

## Task 2: This is the title of task 2

This section should contain the solution of task 2.

## Results and Discussion

This section should contain:
- Results.
- Summary of best model performance:
    - Name of best model file as saved in /models.
    - Relevant scores such as: accuracy, precision, recall, F1-score, etc.
- Key discussion points.

In [ ]:
# Always use comments in the code to document specific steps